In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [3]:
DATA_PATH = "../data/raw/students.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (2000, 17)


,Student_ID,Age,Gender,Year_of_Study,Sleep_Hours,Study_Hours_Per_Week,GPA,Social_Media_Hours,Screen_Time_Hours,Physical_Activity,Financial_Stress,Family_Support,Attendance_Rate,Part_Time_Job,Diet_Quality,Previous_Mental_Health_History,Mental_Health_Risk
0,STU0001,22,Male,3,6.6,17.6,2.91,6.9,7.5,Moderate,8.5,4.0,78.0,No,Poor,No,High
1,STU0002,21,Male,4,5.6,22.4,2.92,7.6,11.5,Low,4.5,5.3,85.4,Yes,Average,No,Medium
2,STU0003,22,Male,5,7.1,28.1,2.68,6.5,8.9,Low,4.8,6.1,79.5,No,Average,No,Medium
3,STU0004,24,Male,1,4.5,14.8,2.23,5.5,7.6,High,3.7,8.0,73.9,No,Average,Yes,High
4,STU0005,20,Female,1,6.7,17.2,2.69,4.3,7.0,Moderate,8.0,7.4,99.8,Yes,NaN,No,Medium


In [4]:
missing_values = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percentage": (df.isnull().sum() / len(df)) * 100
})

print(missing_values['Missing_Count'].sum())

missing_values = missing_values[missing_values["Missing_Count"] > 0]

missing_values = missing_values.sort_values(
    by="Missing_Percentage",
    ascending=False
)

display(missing_values)

419


,Missing_Count,Missing_Percentage
Financial_Stress,91,4.55
Diet_Quality,69,3.45
Sleep_Hours,62,3.10
Social_Media_Hours,58,2.90
Family_Support,54,2.70
Attendance_Rate,47,2.35
GPA,38,1.90


In [5]:
rows_with_missing = df.isnull().any(axis=1).sum()

print("Total rows:", len(df))
print("Rows with missing values:", rows_with_missing)

print(
    "Percentage:",
    round(rows_with_missing / len(df) * 100, 2),
    "%"
)

Total rows: 2000
Rows with missing values: 394
Percentage: 19.7 %


In [6]:
missing_per_row = df.isnull().sum(axis=1)

display(missing_per_row.value_counts().sort_index())

0    1606
1     369
2      25
Name: count, dtype: int64

In [7]:
numerical_columns = df.select_dtypes(include=["number"]).columns


print(numerical_columns)

Index(['Age', 'Year_of_Study', 'Sleep_Hours', 'Study_Hours_Per_Week', 'GPA',
       'Social_Media_Hours', 'Screen_Time_Hours', 'Financial_Stress',
       'Family_Support', 'Attendance_Rate'],
      dtype='str')


In [8]:
outlier_results = []

for column in numerical_columns:

    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    ][column]

    outlier_results.append({
        "Feature": column,
        "Lower_Bound": lower_bound,
        "Upper_Bound": upper_bound,
        "Outlier_Count": len(outliers),
        "Outlier_Percentage": len(outliers) / len(df) * 100
    })

outlier_df = pd.DataFrame(outlier_results)

display(outlier_df)

,Feature,Lower_Bound,Upper_Bound,Outlier_Count,Outlier_Percentage
0,Age,15.5000,27.5000,4,0.20
1,Year_of_Study,-3.5000,8.5000,0,0.00
2,Sleep_Hours,3.6000,10.0000,2,0.10
3,Study_Hours_Per_Week,4.0000,36.0000,7,0.35
4,GPA,1.6150,3.4950,9,0.45
5,Social_Media_Hours,3.0125,8.9125,20,1.00
6,Screen_Time_Hours,3.9500,13.1500,7,0.35
7,Financial_Stress,0.1500,11.7500,0,0.00
8,Family_Support,0.7500,12.3500,0,0.00
9,Attendance_Rate,65.7500,104.5500,12,0.60


In [9]:
for column in numerical_columns:

    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outlier_mask = (
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    )

    outliers = df.loc[outlier_mask, column].sort_values()

    if not outliers.empty:
        print(f"\n{column}")
        print(outliers.to_list())


Age
[28, 28, 28, 29]

Sleep_Hours
[3.3, 3.6]

Study_Hours_Per_Week
[2.4, 2.7, 3.2, 3.6, 3.9, 37.5, 38.3]

GPA
[1.5, 1.6, 3.51, 3.51, 3.53, 3.57, 3.6, 3.64, 3.66]

Social_Media_Hours
[2.1, 2.4, 2.4, 2.8, 2.9, 3.0, 9.0, 9.0, 9.0, 9.0, 9.1, 9.2, 9.2, 9.3, 9.4, 9.4, 9.4, 9.5, 10.0, 10.0]

Screen_Time_Hours
[3.3, 3.4, 3.8, 3.8, 3.8, 13.6, 14.0]

Attendance_Rate
[54.4, 61.1, 61.8, 61.9, 62.4, 62.5, 62.5, 62.8, 63.2, 64.4, 65.3, 65.4]


In [10]:
print("Invalid ages:", ((df["Age"] <= 0) | (df["Age"] > 100)).sum())

print("Invalid sleep hours:", (
    (df["Sleep_Hours"] < 0) | (df["Sleep_Hours"] > 24)
).sum())

print("Invalid study hours:", (
    (df["Study_Hours_Per_Week"] < 0) |
    (df["Study_Hours_Per_Week"] > 168)
).sum())

print("Invalid social media hours:", (
    (df["Social_Media_Hours"] < 0) |
    (df["Social_Media_Hours"] > 24)
).sum())

print("Invalid screen time:", (
    (df["Screen_Time_Hours"] < 0) |
    (df["Screen_Time_Hours"] > 24)
).sum())

print("Invalid attendance:", (
    (df["Attendance_Rate"] < 0) |
    (df["Attendance_Rate"] > 100)
).sum())

invalid_social_media_hours = df[
    df["Social_Media_Hours"] > df["Screen_Time_Hours"]
]

print("Invalid social media hours:", len(invalid_social_media_hours))

df.loc[df['Social_Media_Hours'] > df['Screen_Time_Hours'], ['Screen_Time_Hours', 'Social_Media_Hours']]   

Invalid ages: 0
Invalid sleep hours: 0
Invalid study hours: 0
Invalid social media hours: 0
Invalid screen time: 0
Invalid attendance: 0
Invalid social media hours: 37


,Screen_Time_Hours,Social_Media_Hours
78,4.9,5.0
168,5.3,5.6
209,7.3,7.5
278,5.1,6.0
382,4.1,5.3
424,4.4,5.5
476,5.3,5.5
505,5.5,5.6
557,4.4,4.6
588,5.3,5.7


In [11]:
df = df.drop(index=invalid_social_media_hours.index)

In [12]:
missing_numerical_columns = [
    "Financial_Stress",
    "Sleep_Hours",
    "Social_Media_Hours",
    "Family_Support",
    "Attendance_Rate",
    "GPA"
]

display(
    df[missing_numerical_columns].describe().T
)

,count,mean,std,min,25%,50%,75%,max
Financial_Stress,1872.0,5.961165,1.886454,1.4,4.60,6.10,7.40,9.90
Sleep_Hours,1904.0,6.759296,1.130287,3.3,6.00,6.80,7.60,10.00
Social_Media_Hours,1905.0,6.014488,1.156603,2.1,5.20,6.00,6.70,10.00
Family_Support,1909.0,6.519225,1.840820,1.7,5.10,6.70,8.00,9.90
Attendance_Rate,1917.0,85.057851,7.227824,54.4,80.20,85.10,89.90,100.00
GPA,1926.0,2.561521,0.344055,1.5,2.32,2.56,2.79,3.66


In [13]:
print(df["Diet_Quality"].value_counts(dropna=False))

print("\nPercentages:")
print(
    df["Diet_Quality"]
    .value_counts(normalize=True, dropna=False) * 100
)

Diet_Quality
Average    846
Poor       571
Good       479
NaN         67
Name: count, dtype: int64

Percentages:
Diet_Quality
Average    43.097300
Poor       29.088130
Good       24.401426
NaN         3.413143
Name: proportion, dtype: float64


In [14]:
categorical_columns = df.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

categorical_columns = [
    col for col in categorical_columns
    if col not in ["Student_ID", "Mental_Health_Risk"]
]

for column in categorical_columns:
    print(f"\n{column}")
    print(df[column].value_counts(dropna=False))


Gender
Gender
Female        950
Male          919
Non-binary     94
Name: count, dtype: int64

Physical_Activity
Physical_Activity
Moderate    883
Low         672
High        408
Name: count, dtype: int64

Part_Time_Job
Part_Time_Job
No     1272
Yes     691
Name: count, dtype: int64

Diet_Quality
Diet_Quality
Average    846
Poor       571
Good       479
NaN         67
Name: count, dtype: int64

Previous_Mental_Health_History
Previous_Mental_Health_History
No     1383
Yes     580
Name: count, dtype: int64


C:\Users\ycode\AppData\Local\Temp\ipykernel_26584\1440588468.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(


<h1>feature engineering</h1>

In [15]:
df["Other_Screen_Time"] = (
    df["Screen_Time_Hours"] - df["Social_Media_Hours"]
)

df["Other_Screen_Time"].describe()

count    1905.000000
mean        2.596063
std         1.151556
min         0.000000
25%         1.800000
50%         2.600000
75%         3.400000
max         6.600000
Name: Other_Screen_Time, dtype: float64

In [17]:
OUTPUT_PATH = os.path.join(
    "..", "data", "processed", "students_feature_engineered.csv"
)

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

df.to_csv(OUTPUT_PATH, index=False)